# mandatehub — quickstart (free, no wallet needed)

[mandatehub](https://github.com/Hiroshi-Ichiyanagi/mandatehub) is a live, machine-payable service over **x402** (HTTP 402, real USDC on Base). Each call buys canonically-hashed JSON plus an on-chain settlement tx and a `ProofOfMandate`, for ~0.01 USDC.

This notebook uses only the **free** endpoints — discovery and 402 previews — so it runs anywhere with no key and spends nothing. The last cell shows the one line that would actually pay.

> Open in Colab, then Runtime → Run all.

In [ ]:
BASE = "https://mandatehub.obolpay.xyz"
import json, urllib.request
UA = {"User-Agent": "mandatehub-quickstart/1.0"}
def get(path, headers=None):
    h = dict(UA); h.update(headers or {})
    req = urllib.request.Request(BASE + path, headers=h)
    try:
        with urllib.request.urlopen(req, timeout=60) as r:
            return r.status, json.load(r)
    except urllib.error.HTTPError as e:
        return e.code, json.load(e)

## 1. Discover the catalog (free)

The service self-describes at `/.well-known/agents.json` — every product, its price, and how to pay. It's generated from the live catalog, so it's never out of date.

In [ ]:
_, cat = get("/.well-known/agents.json")
print(cat["summary"], "\n")
print(f"payment: {cat['payment']['protocol']} on {cat['payment']['network']} — "
      f"{cat['payment']['price_minor_units']} minor units (0.01 USDC) per call\n")
for p in cat["products"]:
    print(f"  {p['id']:<14} {p['path']:<28} {p['description'][:60]}")

## 2. Preview a product's payment terms (free)

GET any product URL and you get **HTTP 402** with an `accepts` challenge — the price, network, asset, and recipient. Nothing is charged. (A product that can't be served returns **503** first — also free.)

In [ ]:
status, body = get("/quote")
print("HTTP", status)
if status == 402:
    a = body["accepts"][0]
    print("pay:", a.get("maxAmountRequired") or a.get("amount"), "minor units of USDC")
    print("network:", a["network"])
    print("asset:", a["asset"])
    print("pay_to:", a.get("payTo") or a.get("pay_to"))
elif status == 503:
    print("temporarily unavailable — you would not be charged")

## 3. See the OpenAPI spec (free)

For codegen or an LLM tool schema, `/openapi.json` describes every path, including the per-path `x-402-payment` terms.

In [ ]:
_, spec = get("/openapi.json")
print(spec["info"]["title"], spec["info"]["version"])
print("paths:", list(spec["paths"]))

## 4. Actually pay (optional — spends real USDC)

To buy the data you sign an x402 `exact` payment and resend it in the `X-PAYMENT` header. The `mandatehub` client library does this for you. This needs a **funded Base wallet**, so it's left commented out:

```python
# !pip install 'mandatehub[evm]'
# import os; os.environ['MANDATEHUB_AGENT_PRIVATE_KEY'] = '0x...'  # a Base wallet with USDC
# from mandatehub.x402 import ExactEvmPayloadBuilder, X402PaymentRequirements, encode_x_payment
# from mandatehub.signers import EthAccountSigner
# signer = EthAccountSigner(os.environ['MANDATEHUB_AGENT_PRIVATE_KEY'])
# reqs = X402PaymentRequirements.from_wire(body['accepts'][0])
# header = encode_x_payment(ExactEvmPayloadBuilder(signer, network=reqs.network).build(reqs))
# status, paid = get('/quote', {'X-PAYMENT': header})
# print(paid['data'], paid['settlement']['transaction'], paid['proofOfMandate'])
```

For agents, the `examples/mcp_server.py` (MCP) and `examples/agent_tools.py` (LangChain / CrewAI) wrappers expose exactly this as `discover` / `preview` / `purchase` tools.